# Track A — Bước A2: Nâng cấp Backbone lên `DINOv2-with-registers` (`facebook/dinov2-with-registers-base`) cho `GC10-DET`

- Nhánh: `duytung/06-gc10-backbone-dinov2`
- Mục tiêu: Thay `DINOv1` (`facebook/dino-vitb16`, 2021) bằng **`DINOv2-with-registers`** (`facebook/dinov2-with-registers-base`, 768d, patch 14×14, có 4 register tokens giúp giảm nhiễu nền thép) với cùng `FT_EPOCHS=20`, `r0`, 5 chuỗi `s0..s4`.
- Tự động đo **CPU % peak/avg, RAM peak GB, GPU % peak, GPU VRAM peak MB** và thời gian từng công đoạn, xuất ra `gc10_a2_summary.txt` / `gc10_a2_summary.json` trên Google Drive.
- **Mẹo:** Nếu bạn đang mở sẵn runtime Colab từ Bước A3 (đã mount Drive + đã copy `/content/GC10-DET`), bạn chỉ cần chạy từ Cell 2 trở đi là không phải mount Drive lại!

In [ ]:
# 1. Kết nối Google Drive (bỏ qua nếu runtime hiện tại đã mount sẵn /content/drive)
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Google Drive đã được mount sẵn!')

In [ ]:
# 2. Clone / cập nhật nhánh duytung/06-gc10-backbone-dinov2 lên /content
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'
BRANCH = 'duytung/06-gc10-backbone-dinov2'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin {BRANCH}
!git checkout -B {BRANCH} origin/{BRANCH}
!git status -sb

In [ ]:
# 3. Cài gói thiếu & kiểm tra GPU
!pip uninstall -y -q torchao
!pip install -q numpyro continuum peft datasets transformers scipy scikit-learn psutil
!nvidia-smi
!ls -d /content/GC10-DET /content/drive/MyDrive/Datasets/GC10-DET 2>/dev/null || true

In [ ]:
# 4. Chạy nền Bước A2 (DINOv2-with-registers, FT_EPOCHS=20, r0, s0..s4)
import os, subprocess, time
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a2_dinov2.log'
PID_FILE = f'{WORK_DIR}/gc10_a2_dinov2.pid'

with open(LOG_FILE, 'w') as f:
    p = subprocess.Popen(['bash', 'colab/run_gc10_a2_bg.sh', 'auto'], stdout=f, stderr=subprocess.STDOUT)
with open(PID_FILE, 'w') as f:
    f.write(str(p.pid))
print(f'Đã khởi chạy nền Bước A2 với PID={p.pid}')
print(f'Log: {LOG_FILE}')
time.sleep(3)
!tail -n 20 {LOG_FILE}

In [ ]:
# 5. Kiểm tra tiến độ & xem kết quả + thông số phần cứng peak
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a2_dinov2.log'
PID_FILE = f'{WORK_DIR}/gc10_a2_dinov2.pid'
SUMMARY_OUT = f'{WORK_DIR}/gc10_a2_summary.txt'

pid = open(PID_FILE).read().strip() if os.path.exists(PID_FILE) else ''
alive = False
if pid and os.path.exists(f'/proc/{pid}/status'):
    st = open(f'/proc/{pid}/status').read()
    alive = ('State:\tZ' not in st and 'State: Z' not in st)
print(f'Trạng thái PID={pid}: {"ĐANG CHẠY" if alive else "ĐÃ DỪNG / XONG"}')
print('\n--- 40 dòng log gần nhất ---')
if os.path.exists(LOG_FILE):
    lines = open(LOG_FILE).read().splitlines()
    print('\n'.join(lines[-40:]))
if os.path.exists(SUMMARY_OUT):
    print('\n--- BẢNG KẾT QUẢ & THÔNG SỐ PHẦN CỨNG PEAK BƯỚC A2 (GC10 DINOv2-reg) ---')
    print(open(SUMMARY_OUT).read())